In [ ]:
#!/usr/bin/env python3
"""plot_tpcc_std_panels.py — every subfigure as its own file (for LaTeX subfigures).

    python3 TPC-C/plot_tpcc_std_panels.py OUT_DIR DIR_s42 DIR_s43 DIR_s44 [--width 3.4 --height 2.5]

  In Jupyter, run the cell (defines the functions), then in a new cell:
    main(["final_panels", "results_tpccstd_w8_k4_final",
          "results_tpccstd_w8_k4_s43_final", "results_tpccstd_w8_k4_s44_final",
          "--alg3", "results_tpccstd_w8_elastic/s42", "results_tpccstd_w8_elastic/s43",
          "results_tpccstd_w8_elastic/s44", "--labels"])

--alg3 DIR...  results of run_tpcc_elastic.sh (Max-Threshold, Alg. 3, K decided by
               the threshold; K = 8 here).  Only its SidePart data-aware runs are
               used, drawn as their own line "SidePart data-aware (Alg. 3)".
--systems ...  which lines to draw (default: the four of the paper:
               sidepart_dp_syn sidepart_dp_syn_alg3 sidepart_random obladi).
               Random and Obladi come from the fixed-k (K = 4) directories.

DIR_* are results directories of run_tpcc_std.sh (same systems / rates, one per
seed).  Throughput figures show mean +- sd over the directories; the shard-load
CDFs pool the read rounds of all directories.

Writes to OUT_DIR, each as .pdf and .png:
  latency_vs_tps_all          x = epoch latency, y = committed txn/s, all systems
  latency_vs_tps_sidepart     same, SidePart only, zoomed latency axis
  latency_vs_tpmc_all         y = tpmC
  latency_vs_tpmc_sidepart
  tps_vs_latency_all          x = committed txn/s, y = epoch latency
  tpmc_vs_latency_all
  shard_load_r<rate>          per offered rate: CDF over read rounds of the
                              hottest shard's demand / bread (SidePart systems)
  panels_summary.csv          the numbers behind the throughput figures
Default size is one ACM/VLDB column (3.4 x 2.5 in); fonts are sized for it.
"""
import argparse, csv, glob, os, re, statistics, sys
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
SYSTEMS = [  # key, legend label, colour, marker
    ("sidepart_dp_syn",      "SidePart data-aware (Alg. 2, K=4)", "#2a78d6", "o"),
    ("sidepart_dp_syn_alg3", "SidePart data-aware (Alg. 3, K=8)", "#8f63d6", "D"),
    ("sidepart_dp",          "SidePart data-aware (no synopsis)", "#7fb3ea", "o"),
    ("sidepart_random",      "Random distribution (K=4)",  "#eb6834", "s"),
    ("sidepart_random_syn",  "SidePart random + synopsis", "#c23b7a", "P"),
    ("obladi",               "Obladi",                     "#1baf7a", "^"),
    ("obladi_escrow",        "Obladi + escrow",            "#5c5b55", "v"),
    ("exact_oracle",         "Exact oracle (non-private)", "#eda100", "X"),
]
DEFAULT_SYSTEMS = ["sidepart_dp_syn", "sidepart_dp_syn_alg3", "sidepart_random", "obladi"]
ALG3_KEEP = {"sidepart_dp_syn": "sidepart_dp_syn_alg3"}   # from --alg3 dirs: only this run, renamed
TEXT, MUTED, GRID = "#1f1f1e", "#6b6a63", "#e6e5df"
plt.rcParams.update({"font.size": 8, "axes.labelsize": 8, "legend.fontsize": 7,
                     "xtick.labelsize": 7, "ytick.labelsize": 7, "pdf.fonttype": 42, "ps.fonttype": 42})


In [24]:
def mean_sd(v):
    v = [x for x in v if x == x]
    if not v:
        return float("nan"), 0.0
    return statistics.fmean(v), (statistics.stdev(v) if len(v) > 1 else 0.0)

In [ ]:
def load(dirs, rename=None, runs=None, rounds=None):
    # rename: {system: new key}; systems not in it are skipped (None = keep all as is)
    runs = defaultdict(list) if runs is None else runs       # (system, rate) -> [ {tps, tpmC, epoch_ms} ]
    rounds = defaultdict(list) if rounds is None else rounds # (system, rate) -> [ (bread, [demand per shard]) ]
    for d in dirs:
        d = os.path.expanduser(d)
        if not os.path.isdir(d):
            print(f"  !! not a directory (check the path): {os.path.abspath(d)}")
            continue
        found = glob.glob(os.path.join(d, "*_r*_summary.csv"))
        if not found:   # e.g. a zip unpacked into results_x/results_x/
            found = glob.glob(os.path.join(d, "**", "*_r*_summary.csv"), recursive=True)
        print(f"  {d}: {len(found)} runs")
        for summ in found:
            m = re.match(r"(.+)_r(\d+(?:\.\d+)?)_summary\.csv$", os.path.basename(summ))
            if not m:
                continue
            sysname = m.group(1)
            if rename is not None:
                if sysname not in rename:
                    continue
                sysname = rename[sysname]
            key = (sysname, float(m.group(2)))
            with open(summ) as f:
                s = next(csv.DictReader(f))
            with open(summ.replace("_summary.csv", "_epochs.csv")) as f:
                ms = [float(r["epoch_ms"]) for r in csv.DictReader(f)]
            runs[key].append({"tps": float(s["committed_tps"]), "tpmC": float(s["tpmC"]),
                              "epoch_ms": statistics.fmean(ms) if ms else float("nan")})
            rpath = summ.replace("_summary.csv", "_rounds.csv")
            if os.path.exists(rpath):
                bread = int(float(s.get("bread", 256) or 256))
                per = defaultdict(dict)
                with open(rpath) as f:
                    for r in csv.DictReader(f):
                        per[(r["epoch"], r["round"])][int(r["shard"])] = int(r["real"]) + int(r["rejected"])
                for dem in per.values():
                    if sum(dem.values()) > 0:
                        rounds[key].append((bread, list(dem.values())))
    return runs, rounds

In [26]:
def style(ax):
    ax.grid(True, color=GRID, linewidth=0.6, zorder=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#bdbcb5")
    ax.tick_params(colors=MUTED)

In [27]:
def save(fig, out, name):
    fig.tight_layout(pad=0.3)
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(out, f"{name}.{ext}"), dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("  wrote", name + ".pdf/.png")

In [ ]:
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("out")
    ap.add_argument("dirs", nargs="+")
    ap.add_argument("--width", type=float, default=3.4)
    ap.add_argument("--height", type=float, default=2.5)
    ap.add_argument("--labels", action="store_true", help="print the offered rate next to each point")
    ap.add_argument("--alg3", nargs="*", default=[],
                    help="results dirs of run_tpcc_elastic.sh (Alg. 3, Max-Threshold); data-aware runs only")
    ap.add_argument("--systems", nargs="+", default=DEFAULT_SYSTEMS, help="lines to draw, by key")
    a = ap.parse_args(argv)   # argv=None -> command line; in Jupyter pass a list
    a.out = os.path.expanduser(a.out)
    runs, rounds = load(a.dirs)
    if a.alg3:
        print("  Alg. 3 (Max-Threshold) runs:")
        load(a.alg3, rename=ALG3_KEEP, runs=runs, rounds=rounds)
    if not runs:
        print("!! no *_summary.csv found in", [os.path.abspath(d) for d in a.dirs])
        print("   current directory:", os.getcwd())
        return
    os.makedirs(a.out, exist_ok=True)
    print("  output ->", os.path.abspath(a.out))
    order = [s[0] for s in SYSTEMS]
    present = [s for s in SYSTEMS if s[0] in a.systems and any(k[0] == s[0] for k in runs)]
    missing = [k for k in a.systems if k not in [s[0] for s in present]]
    if missing:
        print("  !! no runs for", missing)

    agg = {}                                        # (system, rate) -> dict of mean/sd
    for key, rr in runs.items():
        agg[key] = {f: mean_sd([r[f] for r in rr]) for f in ("tps", "tpmC", "epoch_ms")}
        agg[key]["n"] = len(rr)
    with open(os.path.join(a.out, "panels_summary.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["system", "offered_txn_s", "n_seeds", "tps_mean", "tps_sd", "tpmC_mean", "tpmC_sd",
                    "epoch_ms_mean", "epoch_ms_sd"])
        for key in sorted((k for k in agg if k[0] in a.systems), key=lambda k: (order.index(k[0]) if k[0] in order else 99, k[1])):
            g = agg[key]
            w.writerow([key[0], key[1], g["n"], *g["tps"], *g["tpmC"], *g["epoch_ms"]])

    def series(sys_key, metric):
        pts = sorted(k for k in agg if k[0] == sys_key)
        return ([agg[k]["epoch_ms"][0] for k in pts], [agg[k]["epoch_ms"][1] for k in pts],
                [agg[k][metric][0] for k in pts], [agg[k][metric][1] for k in pts], [k[1] for k in pts])

    def throughput_fig(metric, swap, only, name, zoom):
        fig, ax = plt.subplots(figsize=(a.width, a.height))
        lat_all = []
        for i, (key, label, color, marker) in enumerate(present):
            if only and key not in only:
                continue
            lx, lxe, ty, tye, rates = series(key, metric)
            lat_all += lx
            xs, xe, ys, ye = (lx, lxe, ty, tye) if swap else (ty, tye, lx, lxe)
            ax.errorbar(xs, ys, xerr=xe, yerr=ye, color=color, marker=marker, markersize=4.5,
                        linewidth=1.4, markeredgecolor="white", markeredgewidth=0.8, capsize=1.8,
                        elinewidth=0.8, label=label, zorder=3)
            if a.labels:
                off = (-14, 3) if i % 2 == 0 else (5, -8)
                for x, y, r in zip(xs, ys, rates):
                    ax.annotate(f"{r:g}", (x, y), textcoords="offset points", xytext=off,
                                fontsize=5.5, color=color)
        tl = "Committed throughput (tpmC)" if metric == "tpmC" else "Committed throughput (txn/s)"
        ll = "Epoch latency (ms)"
        ax.set_xlabel(ll if swap else tl, color=TEXT)
        ax.set_ylabel(tl if swap else ll, color=TEXT)
        style(ax)
        if zoom and lat_all:
            pad = 0.04 * (max(lat_all) - min(lat_all) + 1)
            lo, hi = min(lat_all) - pad - 40, max(lat_all) + pad + 40
            if swap:
                ax.set_xlim(lo, hi); ax.set_ylim(bottom=0)
            else:
                ax.set_ylim(lo, hi); ax.set_xlim(left=0)
        else:
            ax.set_xlim(left=0); ax.set_ylim(bottom=0)
        if zoom:
            ax.legend(frameon=False, loc="best")
        else:   # all systems: legend above the axes (2 columns) so it never covers a line
            ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2,
                      columnspacing=1.0, handlelength=1.8, fontsize=6.5)
        save(fig, a.out, name)

    sidepart = [k for k, *_ in SYSTEMS if k.startswith("sidepart")]
    for metric, tag in (("tps", "tps"), ("tpmC", "tpmc")):
        throughput_fig(metric, True,  None,     f"latency_vs_{tag}_all", False)
        throughput_fig(metric, True,  sidepart, f"latency_vs_{tag}_sidepart", True)
        throughput_fig(metric, False, None,     f"{tag}_vs_latency_all", False)

    # Shard-load CDFs, one file per offered rate (SidePart only: k > 1).
    rates = sorted({k[1] for k in rounds})
    for rate in rates:
        fig, ax = plt.subplots(figsize=(a.width, a.height))
        drew = False
        for key, label, color, _ in present:
            rr = rounds.get((key, rate), [])
            if not rr or len(rr[0][1]) < 2:
                continue
            xs = sorted(max(d) / b for b, d in rr)
            ys = [(i + 1) / len(xs) for i in range(len(xs))]
            ax.step(xs, ys, where="post", color=color, linewidth=1.4, label=label)
            drew = True
        if not drew:
            plt.close(fig)
            continue
        ax.axvline(1.0, color=MUTED, linewidth=0.8, linestyle="--")
        ax.set_xlabel("Hottest shard read demand / batch capacity", color=TEXT)
        ax.set_ylabel("CDF over read rounds", color=TEXT)
        ax.set_xlim(left=0); ax.set_ylim(0, 1.02)
        style(ax)
        ax.legend(frameon=False, loc="lower right")
        save(fig, a.out, f"shard_load_r{rate:g}")


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [ ]:
main(["final_panels",
      "results_tpccstd_w8_k4_final",
      "results_tpccstd_w8_k4_s43_final",
      "results_tpccstd_w8_k4_s44_final",
      "--alg3",
      "results_tpccstd_w8_elastic/s42",
      "results_tpccstd_w8_elastic/s43",
      "results_tpccstd_w8_elastic/s44",
      "--labels"])